# 📖 Lab 5: High Availability & Fault Tolerance (Deep Dive)

**Non-functional requirement:** *The system should be highly available.*

With multiple Redis shards, each is a critical component. If a shard dies, what happens to the users on that shard? We need a failure mode strategy and preventive measures.

## The Fundamental Decision

```
Redis shard goes down → ???

Option A: Fail Closed              Option B: Fail Open
─────────────────────              ────────────────────
Reject ALL requests                Allow ALL requests
API goes offline ❌                 No rate limiting ⚠️
Safe, but kills UX                 Available, but unprotected
```

| | Fail Closed | Fail Open |
|-|-------------|-----------|
| **User experience** | 503 errors even when backend healthy | API works, just unprotected |
| **Security** | ✅ No uncontrolled access | ❌ Attackers can flood backend |
| **Cascade risk** | Low — traffic stopped at gate | High — all traffic hits backend |
| **Best for** | Social media during viral events, payments | Low-risk APIs, internal services |

**We choose: Fail closed** — rate limiter failures often coincide with traffic spikes. Failing open during a viral event → backend flood → cascading collapse.

## Learning Objectives

- Implement fail-open and fail-closed modes
- Build a circuit breaker that detects Redis failures
- Simulate shard failure and see both failure modes in action
- Understand Redis master-replica failover

## 🛠️ Setup

```bash
cd system-designs/rate-limiter
docker-compose up -d
```

Select the **"Rate Limiter (Python)"** kernel.

In [ ]:
import redis
import time
from enum import Enum

redis_client = redis.Redis(host="localhost", port=6381, decode_responses=True)
redis_client.flushdb()

TOKEN_BUCKET_LUA = """
local key = KEYS[1]
local capacity = tonumber(ARGV[1])
local refill_rate = tonumber(ARGV[2])
local now = tonumber(ARGV[3])
local ttl = tonumber(ARGV[4])
local tokens = tonumber(redis.call('HGET', key, 'tokens') or capacity)
local last_refill = tonumber(redis.call('HGET', key, 'last_refill') or now)
local elapsed = now - last_refill
tokens = math.min(capacity, tokens + elapsed * refill_rate)
local allowed = 0
local remaining = math.floor(tokens)
if tokens >= 1 then
    tokens = tokens - 1
    allowed = 1
    remaining = math.floor(tokens)
end
redis.call('HSET', key, 'tokens', tostring(tokens))
redis.call('HSET', key, 'last_refill', tostring(now))
redis.call('EXPIRE', key, ttl)
return {allowed, remaining}
"""
token_bucket_script = redis_client.register_script(TOKEN_BUCKET_LUA)

print("✅ Redis connected, Lua script registered.")

## 🔧 Building a Circuit Breaker

A circuit breaker wraps the Redis call and detects failures. After N consecutive failures, it "opens" the circuit — stopping Redis calls and applying the chosen failure mode directly. This prevents hammering a dead Redis with wasted connection attempts.

```
Closed (normal):     Redis call → success → track
                     Redis call → fail → increment failure count

Open (triggered):    Skip Redis entirely → apply failure mode
                     After cooldown → try one probe request (half-open)

Half-open (probing): Redis call → success → close circuit ✅
                     Redis call → fail → stay open
```

In [ ]:
class CircuitState(Enum):
    CLOSED = "closed"       # Normal — Redis calls work
    OPEN = "open"           # Tripped — Redis is down, skip calls
    HALF_OPEN = "half_open" # Probing — test if Redis is back


class CircuitBreaker:
    """
    Circuit breaker for Redis connections.
    Opens after failure_threshold consecutive failures.
    Probes after cooldown_seconds to check recovery.
    """

    def __init__(self, failure_threshold: int = 3, cooldown_seconds: float = 5.0):
        self.failure_threshold = failure_threshold
        self.cooldown_seconds = cooldown_seconds
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0.0

    def can_execute(self) -> bool:
        if self.state == CircuitState.CLOSED:
            return True

        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.cooldown_seconds:
                self.state = CircuitState.HALF_OPEN
                return True  # Allow one probe request
            return False

        # HALF_OPEN: allow the probe
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = CircuitState.CLOSED

    def record_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.time()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN


class FailureMode(Enum):
    FAIL_OPEN = "fail_open"     # Allow all when Redis down
    FAIL_CLOSED = "fail_closed" # Reject all when Redis down


class ResilientRateLimiter:
    """Rate limiter with circuit breaker and configurable failure mode."""

    def __init__(self, failure_mode: FailureMode, capacity: int = 10, refill_rate: float = 2.0):
        self.failure_mode = failure_mode
        self.capacity = capacity
        self.refill_rate = refill_rate
        self.circuit = CircuitBreaker(failure_threshold=3, cooldown_seconds=5.0)
        self.stats = {"allowed": 0, "rejected": 0, "fallback": 0, "errors": 0}

    def check(self, client_id: str, redis_conn) -> dict:
        """Check rate limit with circuit breaker protection."""

        # Circuit is open — don't even try Redis
        if not self.circuit.can_execute():
            return self._fallback(client_id, reason="circuit_open")

        try:
            key = f"rl:{client_id}"
            result = token_bucket_script(
                keys=[key], args=[self.capacity, self.refill_rate, time.time(), 3600],
                client=redis_conn,
            )
            self.circuit.record_success()

            allowed = bool(result[0])
            if allowed:
                self.stats["allowed"] += 1
            else:
                self.stats["rejected"] += 1

            return {"allowed": allowed, "remaining": result[1], "source": "redis"}

        except (redis.ConnectionError, redis.TimeoutError) as e:
            self.circuit.record_failure()
            self.stats["errors"] += 1
            return self._fallback(client_id, reason="redis_error")

    def _fallback(self, client_id: str, reason: str) -> dict:
        """Apply failure mode when Redis is unreachable."""
        self.stats["fallback"] += 1

        if self.failure_mode == FailureMode.FAIL_OPEN:
            return {"allowed": True, "remaining": -1, "source": f"fallback_open ({reason})"}
        else:
            return {"allowed": False, "remaining": 0, "source": f"fallback_closed ({reason})"}


print("✅ CircuitBreaker + ResilientRateLimiter defined.")

## 🧪 Simulating Redis Failure: Fail Open vs Fail Closed

We'll simulate a Redis failure by connecting to a non-existent Redis port. This lets us see exactly how both failure modes behave without actually killing our Redis container.

In [ ]:
# Simulate: normal requests, then Redis "goes down", then recovers

dead_redis = redis.Redis(host="localhost", port=19999, decode_responses=True, socket_timeout=0.1)
live_redis = redis.Redis(host="localhost", port=6381, decode_responses=True)
live_redis.flushdb()


def simulate_failure_scenario(mode: FailureMode):
    """Simulate: healthy → failure → recovery."""
    rl = ResilientRateLimiter(failure_mode=mode, capacity=5, refill_rate=1.0)

    log = []

    # Phase 1: Normal operation (live Redis)
    for i in range(3):
        result = rl.check("alice", live_redis)
        log.append(("healthy", result))

    # Phase 2: Redis fails (dead connection)
    for i in range(5):
        result = rl.check("alice", dead_redis)
        log.append(("redis_down", result))

    # Phase 3: Recovery (live Redis again, but circuit breaker might be open)
    # Wait for cooldown
    time.sleep(0.1)
    rl.circuit.cooldown_seconds = 0.05  # Speed up for demo
    time.sleep(0.1)

    for i in range(3):
        result = rl.check("alice", live_redis)
        log.append(("recovered", result))

    return log, rl.stats, rl.circuit


# Run both modes
for mode in [FailureMode.FAIL_OPEN, FailureMode.FAIL_CLOSED]:
    print(f"\n{'='*65}")
    print(f"  {'🟢 FAIL OPEN' if mode == FailureMode.FAIL_OPEN else '🔴 FAIL CLOSED'}")
    print(f"{'='*65}\n")

    log, stats, circuit = simulate_failure_scenario(mode)

    print(f"  {'Phase':<14} {'Allowed':<10} {'Remaining':<12} {'Source'}")
    print(f"  {'─'*55}")
    for phase, result in log:
        icon = "✅" if result["allowed"] else "❌"
        print(f"  {phase:<14} {icon:<10} {str(result['remaining']):<12} {result['source']}")

    print(f"\n  📊 Stats: allowed={stats['allowed']}, rejected={stats['rejected']}, "
          f"fallback={stats['fallback']}, errors={stats['errors']}")
    print(f"  Circuit: {circuit.state.value}")

print(f"\n{'='*65}")
print(f"\n💡 Key difference:")
print(f"   Fail OPEN:   During outage, requests are ALLOWED (no protection)")
print(f"   Fail CLOSED: During outage, requests are REJECTED (API offline)")
print(f"   Both recover automatically when Redis comes back.")

## 🏗️ Redis Master-Replica Failover

Prevention is better than fallback. Redis master-replica replication keeps a hot standby ready:

```
Normal:     Gateway ──> Redis Master ──replication──> Redis Replica (standby)

Failure:    Gateway ──> Redis Master ❌
                              ↓ detect failure (sentinel/cluster)
            Gateway ──> Redis Replica → promoted to Master ✅
```

**Redis Cluster** does this automatically:
1. Each master shard has 1+ replicas
2. Cluster nodes continuously health-check each other
3. If a master is unreachable for N seconds, replicas vote to elect a new master
4. Clients are redirected to the new master automatically

**Trade-off:** During the ~1-2 second failover window, some writes may be lost (async replication). For rate limiting, this means a few clients might get slightly more requests through than their limit — acceptable for our eventual consistency requirement.

## 🧹 Cleanup

In [ ]:
live_redis.flushdb()
print("✅ Redis cleaned up.")

## 🏗️ Final Architecture

```
┌────────┐       ┌──────────────────────────────────────────────────┐
│        │       │  API Gateway (N instances)                       │
│        │──────>│                                                  │
│ Client │       │  1. Extract client ID                            │
│        │       │  2. Hash → shard routing                         │
│        │<────  │  3. Token Bucket via Redis Lua (circuit breaker) │
└────────┘       │  4. ✅ 200 or ❌ 429 + headers                   │
                 └──────────────────────┬───────────────────────────┘
                                        │
                          ┌─────────────┼─────────────┐
                          v             v             v
                   ┌────────────┐ ┌────────────┐ ┌────────────┐
                   │Redis Shard1│ │Redis Shard2│ │Redis ShardN│
                   │  Master    │ │  Master    │ │  Master    │
                   │  ↓ replica │ │  ↓ replica │ │  ↓ replica │
                   └────────────┘ └────────────┘ └────────────┘

Circuit breaker per shard:
  Closed → normal Redis calls
  Open → fail-closed (reject all) or fail-open (allow all)
  Half-open → probe to check recovery
```

## ✅ Summary

### Failure Mode Decision Tree

```
Redis unreachable?
  ├── Is this a high-security / high-risk API?
  │   └── YES → Fail CLOSED (reject all, protect backend)
  │
  └── Is brief uncontrolled access acceptable?
      └── YES → Fail OPEN (allow all, keep API available)
```

### What We Built

| Component | Purpose |
|-----------|---------|
| **Circuit Breaker** | Detects Redis failure, stops wasting connection attempts, auto-recovers |
| **Fail-open mode** | Allows requests when Redis down — available but unprotected |
| **Fail-closed mode** | Rejects requests when Redis down — protected but unavailable |
| **Master-replica failover** | Redis Cluster promotes replicas automatically (~1-2s failover) |

### Interview Guidance

- **Acknowledge the trade-off** — fail-open vs fail-closed is a design decision, not a right answer
- **Justify your choice** — connect it to the system's requirements (social media → fail closed during traffic spikes)
- **Mention Redis Cluster failover** — automatic master promotion, ~1-2s window
- **Circuit breaker** — shows you understand operational resilience patterns